# Cleaning the JHU CSSE Global COVID-19 Confirmed Cases Dataset

This notebook transforms the original wide-format time-series CSV into a tidy, long-format table for analysis or Neo4j import. The source is historical and ends in 2023. It reports cases by country/region and, for some countries, province/state. It does not include hospitals or person-to-person transmission links.

Important: keep locations at the source's reported level. Do not sum country totals together with their province rows, since that can double-count cases.

## 1. Load the raw CSV
In Colab, upload `time_series_covid19_confirmed_global.csv` when prompted.

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
raw = pd.read_csv(filename)
print('Rows, columns:', raw.shape)
display(raw.head())
print('Columns:', raw.columns.tolist())

## 2. Inspect the raw data
The first four columns identify the reported location; each remaining column is a cumulative-count date.

In [ ]:
print(raw.dtypes)
display(raw.isna().sum().to_frame('missing_values'))
display(raw.describe(include='all').transpose().head(10))

## 3. Standardize columns and reshape dates into rows
This makes each record one location on one date, which is easier to analyze and import into a graph database.

In [ ]:
# Normalize the known JHU CSSE field names while retaining all date columns.
rename_map = {
    'Province/State': 'province_state',
    'Country/Region': 'country_region',
    'Lat': 'latitude',
    'Long': 'longitude',
}
data = raw.rename(columns=rename_map).copy()
id_cols = ['province_state', 'country_region', 'latitude', 'longitude']
missing_id_cols = [c for c in id_cols if c not in data.columns]
if missing_id_cols:
    raise ValueError(f'Unexpected CSV schema; missing columns: {missing_id_cols}')
date_cols = [c for c in data.columns if c not in id_cols]

tidy = data.melt(
    id_vars=id_cols, value_vars=date_cols,
    var_name='date', value_name='confirmed_cases'
)
tidy['date'] = pd.to_datetime(tidy['date'], errors='coerce')
tidy['confirmed_cases'] = pd.to_numeric(tidy['confirmed_cases'], errors='coerce')
tidy['country_region'] = tidy['country_region'].astype('string').str.strip()
tidy['province_state'] = tidy['province_state'].astype('string').str.strip()
tidy['location_name'] = tidy['province_state'].fillna('').str.strip()
tidy.loc[tidy['location_name'].eq(''), 'location_name'] = tidy['country_region']
display(tidy.head())
print('Tidy rows, columns:', tidy.shape)

## 4. Handle missing values and duplicate location-date rows
Rows without a country, valid date, or confirmed count cannot be analyzed and are removed. Missing province names mean the source reported the country/region without a subregion; they are kept and labeled accordingly. Duplicate records at the same reported location and date are combined.

In [ ]:
before = len(tidy)
tidy = tidy.dropna(subset=['country_region', 'date', 'confirmed_cases']).copy()
tidy = tidy[tidy['country_region'].ne('')].copy()
tidy['province_state'] = tidy['province_state'].fillna('')
tidy['location_name'] = tidy['location_name'].fillna(tidy['country_region'])

key = ['country_region', 'province_state', 'date']
duplicate_rows = int(tidy.duplicated(key, keep=False).sum())
tidy = (tidy.groupby(key, as_index=False, dropna=False)
        .agg(latitude=('latitude', 'first'), longitude=('longitude', 'first'),
             location_name=('location_name', 'first'),
             confirmed_cases=('confirmed_cases', 'sum')))
tidy['confirmed_cases'] = tidy['confirmed_cases'].astype('int64')
print(f'Removed {before - len(tidy)} incomplete rows; rows involved in duplicate keys before aggregation: {duplicate_rows}')
print('Remaining duplicate keys:', tidy.duplicated(key).sum())
display(tidy.head())

## 5. Validate counts and derive daily changes
The source values are cumulative counts. Negative daily changes may occur because historical totals were revised; preserve and flag them instead of silently changing the source data.

In [ ]:
print('Negative cumulative values:', int((tidy['confirmed_cases'] < 0).sum()))
print('Missing coordinates:', int(tidy[['latitude', 'longitude']].isna().any(axis=1).sum()))

tidy = tidy.sort_values(['country_region', 'province_state', 'date']).reset_index(drop=True)
tidy['new_confirmed_cases'] = (
    tidy.groupby(['country_region', 'province_state'])['confirmed_cases'].diff()
)
tidy['revision_or_negative_change'] = tidy['new_confirmed_cases'].lt(0)
display(tidy[tidy['revision_or_negative_change']].head())
display(tidy.tail())

## 6. Export the cleaned data
This CSV is suitable for analysis or a simple Neo4j import. Location rows retain the source's reporting level, so province-level entries are not combined with country-level entries.

In [ ]:
output_file = 'covid19_confirmed_cleaned.csv'
tidy.to_csv(output_file, index=False)
print('Saved:', output_file, '| rows:', len(tidy))
files.download(output_file)